# 🧪 实验一：环境准备与数据探索

**MobileNetV3 图像分类实战教程**

---

## 实验目标

1. 检查 Python/PyTorch 深度学习环境是否配置正确
2. 了解 Tiny ImageNet 数据集的结构与内容
3. 可视化数据集中样本图像及其标签
4. 理解数据预处理和数据加载流程

---


In [ ]:
# ====== 1. 导入所需库 ======
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt
import numpy as np
import os

# 导入 torch_npu（Ascend NPU 支持）
import torch_npu

print(f"PyTorch 版本: {torch.__version__}")
print(f"Torchvision 版本: {torchvision.__version__}")

npu_available = torch.npu.is_available()
print(f"NPU 可用: {npu_available}")
if npu_available:
    print(f"NPU 名称: {torch.npu.get_device_name(0)}")
    print(f"NPU 数量: {torch.npu.device_count()}")
print(f"当前工作目录: {os.getcwd()}")

import warnings
warnings.filterwarnings("ignore", message=".*owner does not match.*")
warnings.filterwarnings("ignore", message=".*TASK_QUEUE_ENABLE.*")
warnings.filterwarnings("ignore", message=".*Permission mismatch.*")

---
## 2. Tiny ImageNet 数据集介绍

**Tiny ImageNet** 是 ImageNet 数据集的缩小版本，常用于教学和快速实验：

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">属性</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">数值</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">类别数</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">200</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">训练集大小</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">100,000 张（每类 500 张）</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">验证集大小</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">10,000 张（每类 50 张）</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">图像尺寸</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">64×64 像素（原始）→ 224×224（训练时）</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">类别标签</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">WordNet Synset ID（如 n02124075）</td>
    </tr>
  </tbody>
</table>

让我们先探索数据集的文件结构。

---
## 2.1 下载 Tiny ImageNet 数据集

如果数据集尚未下载，可以通过以下方式获取。Tiny ImageNet 由 Stanford CS231N 课程提供，包含 200 个类别共 10 万张训练图像。

> ⏳ 数据集大小约 **237 MB**（压缩包），下载和解压可能需要几分钟，请耐心等待。

**下载方式：**
- 自动下载（推荐）：运行下方代码，自动从 Stanford 官方源下载
- 手动下载：访问 [Tiny ImageNet 官方页面](https://tiny-imagenet.herokuapp.com/) 或直接使用 [Stanford 镜像](http://cs231n.stanford.edu/tiny-imagenet-200.zip)

In [ ]:
# ====== 下载并解压 Tiny ImageNet 数据集 ======
import urllib.request
import tarfile
import zipfile
import time
import shutil

# 数据集保存到 notebooks 目录内
dataset_dir = os.path.abspath('tiny-imagenet-200')
parent_dir = os.path.dirname(dataset_dir)  # notebooks 目录
tar_path = os.path.join(parent_dir, 'tiny-imagenet-200.tar.gz')
zip_path = os.path.join(parent_dir, 'tiny-imagenet-200.zip')

# Tiny ImageNet 下载地址（Stanford CS231N）
# 如果官方源不可用，可替换为镜像地址
URLS = [
    "http://cs231n.stanford.edu/tiny-imagenet-200.zip",   # 官方源（ZIP 格式）
    "https://tiny-imagenet.herokuapp.com/tiny-imagenet-200.zip",  # 备用源
]


def download_tiny_imagenet(save_path):
    """尝试从多个源下载 Tiny ImageNet 数据集"""
    for url in URLS:
        print(f"正在尝试下载: {url}")
        print("  这可能需要几分钟...")
        try:
            urllib.request.urlretrieve(url, save_path, reporthook=lambda block, blocksize, total: (
                print(f"  下载进度: {min(100, block * blocksize * 100 // (total or 1)):.0f}%", end="\r"),
            )[-1] if total > 0 else None)
            print(f"\n✅ 下载完成! 文件保存至: {save_path}")
            return True
        except Exception as e:
            print(f"  下载失败: {e}")
            print("  尝试下一个源...\n")
            continue
    return False


def extract_zip(zip_path, extract_to):
    """解压 ZIP 文件"""
    print("正在解压 ZIP 文件...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_to)
    print("解压完成!")


def extract_tar(tar_path, extract_to):
    """解压 tar.gz 文件"""
    print("正在解压 tar.gz 文件...")
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(path=extract_to)
    print("解压完成!")


def organize_validation_set(dataset_dir):
    """
    整理验证集为 ImageFolder 格式：
    val_annotations.txt 中的映射将 val/images/ 下的图片
    复制到 val/<类别ID>/ 目录下
    """
    val_dir = os.path.join(dataset_dir, 'val')
    val_images_dir = os.path.join(val_dir, 'images')
    val_annotations = os.path.join(val_dir, 'val_annotations.txt')

    if not os.path.exists(val_annotations):
        print("  ⚠️ 未找到 val_annotations.txt，跳过验证集整理")
        return

    print("  正在整理验证集...")
    with open(val_annotations, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 2:
                continue
            img_name, class_id = parts[0], parts[1]
            src = os.path.join(val_images_dir, img_name)
            dst_dir = os.path.join(val_dir, class_id)
            os.makedirs(dst_dir, exist_ok=True)
            dst = os.path.join(dst_dir, img_name)
            if os.path.exists(src) and not os.path.exists(dst):
                shutil.copy2(src, dst)

    # 清理旧的 images 目录和注释文件
    if os.path.exists(val_images_dir):
        shutil.rmtree(val_images_dir)
        print("    已删除旧的 images/ 目录")
    if os.path.exists(val_annotations):
        os.remove(val_annotations)
        print("    已删除 val_annotations.txt")

    # 验证结果
    val_class_count = len([d for d in os.listdir(val_dir) if os.path.isdir(os.path.join(val_dir, d))])
    val_file_count = sum(len(files) for _, _, files in os.walk(val_dir))
    print(f"  验证集: {val_class_count} 个类别, {val_file_count} 张图片")


# ====== 主流程 ======
if os.path.exists(dataset_dir) and len(os.listdir(dataset_dir)) > 0:
    # 检查数据集是否已存在且非空
    train_dir = os.path.join(dataset_dir, 'train')
    if os.path.exists(train_dir) and len(os.listdir(train_dir)) > 0:
        print(f"✅ 数据集已存在: {dataset_dir}")
        print(f"  训练集类别数: {len([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])}")
    else:
        print(f"⚠️  目录存在但内容不完整，需要重新下载...")
        need_download = True
else:
    need_download = True

if need_download:
    os.makedirs(parent_dir, exist_ok=True)
    start_time = time.time()

    # 优先尝试 ZIP 下载
    success = download_tiny_imagenet(zip_path)

    if success:
        extract_zip(zip_path, parent_dir)
        # 下载完成后删除压缩包（可选，注释掉以保留）
        # os.remove(zip_path)
        # print("已删除压缩包以节省空间")
    else:
        print("⚠️ 自动下载失败，请尝试手动下载:")
        print("  1. 访问 https://tiny-imagenet.herokuapp.com/")
        print("  2. 下载 tiny-imagenet-200.zip")
        print(f"  3. 将文件放入 {parent_dir}/ 目录")
        print(f"  4. 重新运行本单元格")
        # 检查是否有本地 tar.gz 文件
        if os.path.exists(tar_path):
            print("\n检测到本地 tar.gz 文件，尝试解压...")
            extract_tar(tar_path, parent_dir)

    elapsed = time.time() - start_time
    if os.path.exists(dataset_dir):
        print(f"✅ 数据集准备完成! 耗时: {elapsed:.1f} 秒")

# 整理验证集目录结构
print("\n正在整理验证集目录结构...")
organize_validation_set(dataset_dir)

print("\n🎉 数据集下载与准备完成!")

In [ ]:
# ====== 2. 探索数据集结构 ======
data_dir = os.path.abspath('tiny-imagenet-200')

print("数据集目录结构:")
print(f"  根目录: {data_dir}")
print(f"  存在: {os.path.exists(data_dir)}")

if os.path.exists(data_dir):
    # 列出一级目录
    for item in os.listdir(data_dir):
        item_path = os.path.join(data_dir, item)
        if os.path.isdir(item_path):
            # 统计子目录数量
            subdirs = [d for d in os.listdir(item_path) if os.path.isdir(os.path.join(item_path, d))]
            print(f"  └── {item}/    ({len(subdirs)} 个子目录)")
        else:
            size = os.path.getsize(item_path)
            print(f"  └── {item}    ({size:,} 字节)")

In [ ]:
# ====== 3. 查看类别标签 ======
wnids_path = os.path.join(data_dir, 'wnids.txt')
words_path = os.path.join(data_dir, 'words.txt')

# 读取类别ID
with open(wnids_path, 'r') as f:
    class_ids = [line.strip() for line in f.readlines()]
print(f"类别总数: {len(class_ids)}")

# 读取类别名称映射
word_map = {}
with open(words_path, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            word_map[parts[0]] = parts[1]

# 显示前10个类别
print("\n前10个类别:")
for i, cid in enumerate(class_ids[:10]):
    name = word_map.get(cid, 'unknown')
    print(f"  {i+1:3d}. {cid}  →  {name}")

# 统计每类训练样本数
train_dir = os.path.join(data_dir, 'train')
sample_counts = {}
for cid in class_ids[:5]:  # 只看前5类
    img_dir = os.path.join(train_dir, cid, 'images')
    if os.path.exists(img_dir):
        n = len([f for f in os.listdir(img_dir) if f.endswith('.JPEG')])
        sample_counts[cid] = n
        print(f"  类别 {cid} ({word_map.get(cid, '')}): {n} 张训练图像")

---
## 3. 可视化训练样本

让我们从训练集中随机选取一些图像，观察它们的原始尺寸和内容。

In [ ]:
# ====== 4. 可视化训练样本 ======
from PIL import Image

# 选取前5个类别，每个类别显示4张图
fig, axes = plt.subplots(5, 4, figsize=(12, 15))
fig.suptitle('Tiny ImageNet Tiny ImageNet Training Samples', fontsize=16)

for row, cid in enumerate(class_ids[:5]):
    img_dir = os.path.join(train_dir, cid, 'images')
    if not os.path.exists(img_dir):
        continue
    
    # 获取该类别所有图像文件
    img_files = [f for f in os.listdir(img_dir) if f.endswith('.JPEG')][:4]
    
    for col, img_file in enumerate(img_files):
        img_path = os.path.join(img_dir, img_file)
        img = Image.open(img_path)
        
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        axes[row, col].set_title(f'{word_map.get(cid, cid)}', fontsize=9)

plt.tight_layout()
plt.show()

# 打印图像尺寸
print(f"原始图像尺寸: {img.size}")
print(f"训练时将 resize 到: 224×224")

---
## 4. 数据预处理流程

对于 Tiny ImageNet，训练集使用了多种数据增强技术来防止过拟合：

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">增强方法</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">参数</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">作用</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">RandomResizedCrop</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">scale=(0.6, 1.0)</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">随机裁剪并缩放到 224×224，增加尺度多样性</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">RandomHorizontalFlip</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">p=0.5</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">随机水平翻转，增加翻转不变性</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">RandAugment</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">num_ops=2, magnitude=7</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">自动数据增强，组合多种图像变换</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">RandomErasing</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">p=0.1</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">随机擦除图像区域，提高鲁棒性</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">Normalize</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">mean/std</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">标准化到标准正态分布</td>
    </tr>
  </tbody>
</table>

下面我们演示数据预处理的效果。

In [ ]:
# ====== 5. 演示数据增强效果 ======
from torchvision.transforms import RandAugment

# 训练时使用的数据增强 pipeline
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=7),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1),
])

# 验证时使用的预处理（无数据增强）
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 选取一张原始图像
sample_cid = class_ids[0]
sample_img_dir = os.path.join(train_dir, sample_cid, 'images')
sample_files = [f for f in os.listdir(sample_img_dir) if f.endswith('.JPEG')]
original_img = Image.open(os.path.join(sample_img_dir, sample_files[0]))

# 显示原始图像和多个增强版本
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle(f'Data Augmentation Comparison ({word_map.get(sample_cid, sample_cid)})', fontsize=14)

# 原始图像
axes[0, 0].imshow(original_img)
axes[0, 0].set_title('Original (64×64)')
axes[0, 0].axis('off')

# 验证预处理（resize + center crop）
val_img = val_transform(original_img)
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
val_img_display = val_img * std + mean
axes[0, 1].imshow(val_img_display.permute(1, 2, 0).numpy())
axes[0, 1].set_title('Val Preprocess (224×224)')
axes[0, 1].axis('off')

# 6个训练增强版本（分布在2行4列的剩余位置）
positions = [(0, 2), (0, 3), (1, 0), (1, 1), (1, 2), (1, 3)]
for (row, col), _ in zip(positions, range(6)):
    aug_img = train_transform(original_img)
    aug_img_display = aug_img * std + mean
    axes[row, col].imshow(aug_img_display.permute(1, 2, 0).numpy())
    axes[row, col].set_title(f'Augmented {positions.index((row, col)) + 1}')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()
print("💡 可以看到：训练时数据增强显著改变了图像的裁剪区域、颜色和构图，"
      "这有助于模型学习更鲁棒的特征。")

---
## 5. DataLoader 加载数据

使用 `ImageFolder` 和 `DataLoader` 来批量加载数据。

In [ ]:
# ====== 6. 创建 DataLoader 并查看数据形状 ======
batch_size = 32

# 使用验证集预处理（轻量级）
simple_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = ImageFolder(
    os.path.join(data_dir, 'train'),
    transform=simple_transform
)

val_dataset = ImageFolder(
    os.path.join(data_dir, 'val'),
    transform=simple_transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=0
)

val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=0
)

print(f"训练集大小: {len(train_dataset)} 张图像")
print(f"验证集大小: {len(val_dataset)} 张图像")
print(f"类别数: {len(train_dataset.classes)}")
print(f"训练 batch 数: {len(train_loader)}")
print(f"验证 batch 数: {len(val_loader)}")

# 查看一个 batch 的数据形状
data_iter = iter(train_loader)
images, labels = next(data_iter)
print(f"\n图像张量形状: {images.shape}")  # [batch_size, 3, 224, 224]
print(f"标签张量形状: {labels.shape}")    # [batch_size]
print(f"标签范围: {labels.min().item()} ~ {labels.max().item()}")
print(f"图像值范围: {images.min().item():.3f} ~ {images.max().item():.3f}")

---
## 📝 实验小结

在本实验中，我们完成了：

1. ✅ 确认了深度学习环境（PyTorch + NPU）配置正确
2. ✅ 了解了 Tiny ImageNet 数据集的结构：200 个类别，100,000 张训练图像
3. ✅ 可视化了样本图像及其对应的类别名称
4. ✅ 理解了训练/验证阶段的数据预处理流程
5. ✅ 学会了使用 `ImageFolder` 和 `DataLoader` 加载数据

**关键要点：**
- 原始图像是 64×64，训练时会 resize 到 224×224
- 训练集使用 RandAugment、RandomErasing 等数据增强技术
- 验证集只做 resize + center crop + normalize，不做增强
- ImageFolder 会自动按文件夹名称排序分配类别索引

---


## 课后练习

1. (单选题) Tiny ImageNet 有 200 类，训练集每类 500 张、验证集每类 50 张，验证集总数是？
   - A. 10,000
   - B. 50,000
   - C. 100,000
   - D. 20,000

2. (单选题) torchvision.datasets.ImageFolder 的类别索引顺序取决于？
   - A. 子目录名称排序
   - B. val_annotations.txt
   - C. 文件系统返回顺序
   - D. 图片修改时间

3. (单选题) RandomResizedCrop(size=224, scale=(0.08, 1.0), ratio=(3/4, 4/3)) 中 scale 表示？
   - A. 裁剪区域面积占原图面积的比例范围
   - B. 输出尺寸缩放倍数
   - C. 亮度缩放范围
   - D. 通道数

4. (多选题) 以下哪些 transform 会改变输出图像的宽高？
   - A. Resize(256)
   - B. RandomResizedCrop(224)
   - C. CenterCrop(224)
   - D. Normalize

5. (多选题) 关于 RandAugment(num_ops=2, magnitude=9) 的正确说法包括？
   - A. 每个样本从操作集中随机选择 2 个增强操作
   - B. 每个样本使用完全相同的 2 个操作
   - C. magnitude 控制增强强度
   - D. 可包含旋转、平移、对比度等图像级操作

6. (判断题) Tiny ImageNet 原始图片尺寸为 224×224。

7. (判断题) RandomErasing 通常只用于训练集，不应默认加入验证集。

8. (填空题) 验证集按 class 子目录整理后，每类图片数为 ____，验证集总数为 ____。

9. (填空题) transforms.ToTensor() 将像素值缩放到 ____ 区间；transforms.Normalize 使用 ____ 和标准差做标准化。

10. (简答题) 为什么 64×64 的 Tiny ImageNet 通常先 Resize 到 256，再做 CenterCrop 到 224，而不是直接 Resize 到 224？

11. (简答题) 增强强度过大为什么可能降低准确率？请结合语义破坏与分布漂移说明。

12. (代码设计题) 编写 create_transforms(train)，训练时使用 RandomResizedCrop(224)、RandAugment、RandomErasing、ToTensor、Normalize；验证时使用 Resize(256)+CenterCrop(224)+ToTensor+Normalize，并说明验证集为何不用随机增强。

13. (单选题) 训练集与验证集类别分布差异很大时，最可能的影响是？
   - A. 验证指标无法真实反映模型泛化能力
   - B. 训练速度变慢
   - C. 显存升高
   - D. 学习率必须调大

14. (多选题) 验证集准备与评估中必须保持一致的是？
   - A. 与训练相同的 Normalize 参数
   - B. model.eval() 与 torch.no_grad()
   - C. 输入尺寸 224×224
   - D. RandAugment 随机增强

15. (简答题) 设计一种方法快速检查数据增强是否正确：既要看到增强效果，又要确认标签与图片对应关系。

> 参考答案见 answer/02.03_dataset_exploration_answer.ipynb。